<a href="https://colab.research.google.com/github/Abdullah-Farooq292/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdullah-Farooq292/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule combines two signals: staleness (days_since_last_update) and visibility (impressions_90d). The idea: a page that hasn't been updated in a long time, but is still getting real traffic, is a strong candidate for review — it has an audience worth protecting, and the risk of losing that traffic grows the longer it goes unrefreshed.

Rule: stale_and_visible_score = (days_since_last_update >= 180) * (impressions_90d >= 500) * impressions_90d

Reason codes this rule can output:
- "stale_visible_page": days_since_last_update >= 180 AND impressions_90d >= 500 → suggests refresh
- "healthy_page": does not meet the above → no action needed

Verdict: CONFIRMED. Among the 17 pages matching "stale + visible" (days_since_last_update >= 180 and impressions_90d >= 500), 94.1% are declining — compared to only 54.2% among all other pages. This is a strong, real signal, confirming the rule is worth using as a baseline.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, subprocess

if not os.path.isdir("flyrank-ml-internship"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Abdullah-Farooq292/flyrank-ml-internship"], check=True)
os.chdir("flyrank-ml-internship")

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

stale_visible = df[(df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)]
not_stale_visible = df[~((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500))]

print(f"Stale+visible pages: {len(stale_visible)}")
print(f"Decline rate among stale+visible: {(stale_visible['trend_direction']=='down').mean():.3f}")
print(f"Decline rate among everyone else: {(not_stale_visible['trend_direction']=='down').mean():.3f}")

Stale+visible pages: 17
Decline rate among stale+visible: 0.941
Decline rate among everyone else: 0.542


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

This section encodes my confirmed rule into an actual score, assigns a reason code and action label to every page, ranks the full dataset, and writes the result to work/outputs/baseline_action_score.csv.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os

# Re-use df from Section 1 (already loaded)

# Score: stale + visible pages get points proportional to how much traffic they have
df["stale_and_visible_score"] = (
    (df["days_since_last_update"] >= 180).astype(int)
    * (df["impressions_90d"] >= 500).astype(int)
    * df["impressions_90d"]
)

# Reason code per page
def get_reason_code(row):
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        return "stale_visible_page"
    return "healthy_page"

df["reason_code"] = df.apply(get_reason_code, axis=1)

# Action label per page
def get_action(row):
    if row["reason_code"] == "stale_visible_page":
        return "review_for_refresh"
    return "no_action"

df["action"] = df.apply(get_action, axis=1)

# Rank everything by score, descending
ranked = df.sort_values("stale_and_visible_score", ascending=False)

# Make sure the outputs folder exists, then write the CSV
os.makedirs("work/outputs", exist_ok=True)
ranked[["content_id", "stale_and_visible_score", "reason_code", "action",
        "impressions_90d", "days_since_last_update", "trend_direction"]].to_csv(
    "work/outputs/baseline_action_score.csv", index=False
)

print(f"Wrote {len(ranked)} ranked rows to work/outputs/baseline_action_score.csv")
print(ranked[["content_id", "stale_and_visible_score", "reason_code", "action"]].head(10))


Wrote 30000 ranked rows to work/outputs/baseline_action_score.csv
                 content_id  stale_and_visible_score         reason_code  \
16751  content_cf56e2e2e282                    61678  stale_visible_page   
16514  content_7368877ea310                    59472  stale_visible_page   
7021   content_1bfaa38ff26c                    25715  stale_visible_page   
21268  content_0a91db491d14                    13299  stale_visible_page   
11489  content_5feee3994adb                     7812  stale_visible_page   
12045  content_c2d929d83eaa                     7558  stale_visible_page   
698    content_b16bd7307b39                     4590  stale_visible_page   
5327   content_fe16a55cd13d                     4556  stale_visible_page   
26810  content_ecb6215e79fd                     4429  stale_visible_page   
20837  content_928af3e22c80                     1697  stale_visible_page   

                   action  
16751  review_for_refresh  
16514  review_for_refresh  
7021   review

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review: Only 17 pages in the full 30,000-page dataset actually match this rule (score > 0) — the queue runs out of real matches after rank 17, so rows 18-20 (score = 0, reason_code = "healthy_page") are not genuine recommendations, just padding from sorting.

1. content_cf56e2e2e282 — Action: review_for_refresh. Why: 61,678 impressions with no update in 194 days, position 19.7, declining. Highest-traffic stale page in the dataset. What would make it wrong: if this is an intentionally static reference page that doesn't need frequent updates.

2. content_7368877ea310 — Action: review_for_refresh. Why: 59,472 impressions, 194 days stale, declining. What would make it wrong: if position 24.8 is already near a natural ceiling for its content type, refreshing may not move the needle.

3. content_1bfaa38ff26c — Action: review_for_refresh. Why: 25,715 impressions, 194 days stale, CTR 0.23, declining. What would make it wrong: CTR is relatively decent already — the real issue may be position (22.2), not content freshness.

4. content_0a91db491d14 — Action: review_for_refresh. Why: 13,299 impressions, 193 days stale, position 10.5 (near page 1), declining. What would make it wrong: being this close to page 1 with reasonable CTR (0.49) suggests it might just need a small tweak, not a full refresh.

5. content_5feee3994adb — Action: review_for_refresh. Why: 7,812 impressions, 194 days stale, CTR only 0.01, declining. What would make it wrong: extremely low CTR despite decent position (39.0) may point to a title/meta problem, not staleness — refreshing content alone might not fix it.

6. content_c2d929d83eaa — Action: review_for_refresh. Why: 7,558 impressions, 193 days stale, CTR 0.20, declining. What would make it wrong: mid-range signals across the board — this page might be a lower-priority case compared to higher-traffic pages above it.

7. content_b16bd7307b39 — Action: review_for_refresh. Why: 4,590 impressions, 194 days stale, CTR 0.00, declining. What would make it wrong: 0% CTR with real impressions is unusual — worth checking if this is a broken link or indexing issue rather than a content-quality problem.

8. content_fe16a55cd13d — Action: review_for_refresh. Why: 4,556 impressions, 194 days stale, CTR 0.33 (relatively strong), declining. What would make it wrong: a CTR this high suggests the page is actually performing reasonably — the "declining" label alone may not justify urgent refresh.

9. content_ecb6215e79fd — Action: review_for_refresh. Why: 4,429 impressions, 194 days stale, CTR 0.38, declining. What would make it wrong: similar to #8 — decent CTR despite decline suggests the issue may be elsewhere (e.g., seasonality), not staleness.

10. content_928af3e22c80 — Action: review_for_refresh. Why: 1,697 impressions, 193 days stale, declining. What would make it wrong: lower traffic volume means less certainty — could be noise rather than a real pattern.

11. content_e3ff1b093148 — Action: review_for_refresh. Why: 1,408 impressions, 183 days stale (just over threshold), declining. What would make it wrong: barely crosses the staleness threshold — a softer case than the pages above it.

12. content_bdbec75c1148 — Action: review_for_refresh. Why: 1,316 impressions, 194 days stale, but trend_direction is "stable", not declining. What would make it wrong: this page isn't actually declining — it may not need urgent review despite matching the rule.

13. content_7f116ae1f6f5 — Action: review_for_refresh. Why: only 954 impressions but 301 days stale (very old), declining. What would make it wrong: low traffic volume makes this a lower-priority case despite the long staleness.

14. content_77d4d5930e5e — Action: review_for_refresh. Why: 828 impressions, 194 days stale, declining. What would make it wrong: low volume, borderline case for review priority.

15. content_72496874f806 — Action: review_for_refresh. Why: 821 impressions, 301 days stale (long-neglected), declining. What would make it wrong: very low impressions despite long staleness — may not be worth reviewer time compared to higher-traffic pages.

16. content_6226ee6adc91 — Action: review_for_refresh. Why: 545 impressions, 183 days stale, declining. What would make it wrong: low traffic, marginal case.

17. content_074ba6ead17b — Action: review_for_refresh. Why: 533 impressions, 183 days stale, CTR 0.00, declining. What would make it wrong: very low volume and 0% CTR — could be a technical issue (indexing/broken page) rather than a content problem.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = ranked[["content_id", "stale_and_visible_score", "reason_code", "action",
                 "impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]].head(20)
print(top20.to_string())

                 content_id  stale_and_visible_score         reason_code              action  impressions_90d  days_since_last_update  avg_position   ctr trend_direction
16751  content_cf56e2e2e282                    61678  stale_visible_page  review_for_refresh            61678                     194          19.7  0.15            down
16514  content_7368877ea310                    59472  stale_visible_page  review_for_refresh            59472                     194          24.8  0.13            down
7021   content_1bfaa38ff26c                    25715  stale_visible_page  review_for_refresh            25715                     194          22.2  0.23            down
21268  content_0a91db491d14                    13299  stale_visible_page  review_for_refresh            13299                     193          10.5  0.49            down
11489  content_5feee3994adb                     7812  stale_visible_page  review_for_refresh             7812                     194          39.0  0

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: Rows 12-17 in my top-20 are the weakest picks. #12 (content_bdbec75c1148) actually has trend_direction "stable", not "down" — meaning my rule flagged a page that isn't genuinely declining, just because it matches staleness+visibility thresholds. Rows 13-17 also have relatively low impressions (under 1,000), making them lower-confidence cases where the pattern could be closer to noise than a real trend.

Leakage check: My rule uses only days_since_last_update and impressions_90d — both are observable signals known before any decision was made about the page, not derived from the outcome (trend_direction) itself. I did not use trend_direction, trend_pct, or any FlyRank product flag (health_score, priority_score) as an input to the score — those would count as leakage, since they either encode the label directly or represent a decision already made by the product. The score is fully independent of the label I'm trying to predict.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Leakage sanity check — score should NOT be built from label-derived fields
print("Features used in score: days_since_last_update, impressions_90d")
print("Label field (not used as input): trend_direction")
print(f"Correlation between score and impressions_90d (expected, by design): "
      f"{ranked['stale_and_visible_score'].corr(ranked['impressions_90d']):.3f}")


Features used in score: days_since_last_update, impressions_90d
Label field (not used as input): trend_direction
Correlation between score and impressions_90d (expected, by design): 0.028


## Self-check

Before you submit, confirm each line honestly:

- [done ] Every section above is filled — markdown thinking AND the code that backs it
- [done ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [done ] No client names, URLs, or private queries anywhere
- [done ] My claims use careful words: observed, measured, directional, decision-support
- [done ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.